In [ ]:
from pathlib import Path
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive
import pandas as pd
from uncertainties import ufloat

def get_nexsci_data(table_name="ps", clobber=False, outdir="."):
    """
    ps: self-consistent set of parameters
    pscomppars: a more complete, though not necessarily self-consistent set of parameters
    """
    url = "https://exoplanetarchive.ipac.caltech.edu/docs/API_PS_columns.html"
    print("Column definitions: ", url)
    fp = Path(outdir,f"nexsci_{table_name}.csv")
    if not fp.exists() or clobber:
        print(f"Downloading NExSci {table_name} table...")
        nexsci_tab = NasaExoplanetArchive.query_criteria(table=table_name, 
                                                         # where="discoverymethod like 'Transit'"
                                                        )
        df_nexsci = nexsci_tab.to_pandas()
        df_nexsci.to_csv(fp, index=False)
        print("Saved: ", fp)
    else:
        df_nexsci = pd.read_csv(fp)
        print("Loaded: ", fp)
    return df_nexsci

def theoretical_boundary_loglog(period, method='stripe'):
    """
    neptune desert boundary in log space
    """
    if method=='stripe':
        lower = 0.68*period
        upper = -0.33*period+1.17
    elif method=='likelihood':
        from uncertainties import ufloat
        a=ufloat(0.67,0.06)
        b=ufloat(0.01,0.04)
        lower = a*period-b
        
        a=ufloat(0.31,0.12)
        b=ufloat(1.19,0.06)
        upper = -a*period+b
    return lower, upper

def theoretical_boundary(period):
    """
    neptune desert boundary in linear space
    """
    lower = period**0.68
    upper = 14.8*period**-0.33
    return lower, upper

def vaneylen_model_linear(p): 
    """
    period-radius
    
    https://ui.adsabs.harvard.edu/abs/2018MNRAS.479.4786V/abstract
    """
    y1 = p**(-0.09+0.02)*10**(0.37+0.04)
    y2 = p**(-0.09-0.04)*10**(0.37-0.02)
    y = p**-0.09*(10**0.37)
    return y

def vaneylen_model_loglog(x): 
    """
    period-radius: log10R = mlog10P + a
    
    https://ui.adsabs.harvard.edu/abs/2018MNRAS.479.4786V/abstract
    """
    y1 = x*(-0.09+0.02)+(0.37+0.04)
    y2 = x*(-0.09-0.04)+(0.37-0.02)
    return y1, y2

#@|-----------------------------------------------------------------
#@|Boundaries of the Neptunian desert (Castro-González et al. 2024)
#@|-----------------------------------------------------------------

def empirical_boundary_loglog():
    x_tw_low = np.linspace(-0.30, np.log10(2.97), 100)
    x_tw_up = np.linspace(np.log10(2.97),0.120,  100)
    y_tw_up = -0.43 * x_tw_up + 1.14
    y_tw_low = +0.55 * x_tw_low + 0.36
    
    # print(x_tw_low,y_tw_low)
    # print(x_tw_up,y_tw_up)
    
    x1 = np.tile(-0.30,100)
    y1= np.linspace(-0.07,0.191,100)
    
    x2 = np.tile(0.12,100)
    y2 = np.linspace(1.0884,1.20,100)

    x  = np.concatenate([x1,x_tw_low,x_tw_up,x2])
    y = np.concatenate([y1,y_tw_low,y_tw_up,y2])
    return x,y

def empirical_boundary_linear():
    # Generate log10-scaled x values
    x_tw_low_log = np.linspace(-0.30, np.log10(2.97), 100)
    x_tw_up_log = np.linspace(np.log10(2.97), 0.120, 100)

    # Corresponding log10-scaled y values
    y_tw_up_log = -0.43 * x_tw_up_log + 1.14
    y_tw_low_log = +0.55 * x_tw_low_log + 0.36

    # Vertical boundaries in log space
    x1_log = np.tile(-0.30, 100)
    y1_log = np.linspace(-0.07, 0.191, 100)

    x2_log = np.tile(0.12, 100)
    y2_log = np.linspace(1.0884, 1.20, 100)

    # Combine all log-space data
    x_log = np.concatenate([x1_log, x_tw_low_log, x_tw_up_log, x2_log])
    y_log = np.concatenate([y1_log, y_tw_low_log, y_tw_up_log, y2_log])

    # Convert to linear space
    x_linear = 10 ** x_log
    y_linear = 10 ** y_log

    return x_linear, y_linear

# neptune desert boundaries in log space
def upper_boundary_loglog(period, method='stripe'):
    if method=='stripe':
        return -0.33*period+1.17
    elif method=='likelihood':
        a=ufloat(0.31,0.12)
        b=ufloat(1.19,0.06)
        return -a*period+b
    
def lower_boundary_loglog(period, method='stripe'):
    if method=='stripe':
        return 0.68*period
    elif method=='likelihood':
        a=ufloat(0.67,0.06)
        b=ufloat(0.01,0.04)
        return a*period-b

# neptune desert perimeter in linear space
def upper_boundary(period):
    return 14.8*period**-0.33

def lower_boundary(period):
    return period**0.68
    
df_nexsci = get_nexsci_data(table_name="ps", clobber=False)
df_nexsci.head()

In [ ]:
# get values from default references
idx = df_nexsci['default_flag'] == 1
all_hostnames = df_nexsci[idx].hostname.unique()
len(all_hostnames)

In [ ]:
#TOI-1136
idx = df_nexsci[df_nexsci.hostname=="TOI-1136"][df_nexsci[df_nexsci.hostname=="TOI-1136"].pl_refname.str.contains("Dai")].index
df_nexsci.loc[idx,'default_flag']=1

In [ ]:
import astropy.units as u

# get only transiting planets
df_nexsci['pl_rade'] = df_nexsci['pl_radj']*u.Rjup.to(u.Rearth)

# filter data
idx = (df_nexsci.pl_orbper>0) & \
        (df_nexsci.pl_rade>0) & \
        (df_nexsci.st_mass>0) & \
        (df_nexsci.discoverymethod.isin(['Transit','Radial Velocity'])) & \
        (df_nexsci['default_flag'] == 1)
df = df_nexsci[idx]
df.shape

## Ages

In [ ]:
import pandas as pd

df_hosts = pd.read_csv("/ut2/jerome/github/research/project/TOI-6715/data/young_validated_planets_TOIs - validated_hosts.csv", 
                       comment='#', skiprows=[1]).dropna(subset=['Host'],
                                             )
df_hosts = df_hosts.iloc[:68]
df_hosts = df_hosts.reset_index(drop=True)
df_hosts.shape

In [ ]:
# df_hosts['hostname'] = df_hosts.Host.apply(lambda x: x.lower().replace(' ', ''))
df_hosts['hostname'] = df_hosts.Host.str.lower().str.replace(' ', '')
young_hostnames = df_hosts['hostname'].reset_index(drop=True)
young_hostnames

In [ ]:
len(young_hostnames)

In [ ]:
# cross-match NExSci data with own my list based on (edited) hostname
df = df.copy()
df['hostname'] = df['hostname'].str.lower().str.replace(' ', '')
idx = df['hostname'].isin(young_hostnames)
df_planets = df[idx]
df_planets.shape

In [ ]:
#planets in my list not found in NExSci
found=df['hostname'][idx].tolist()
young_hostnames[~young_hostnames.isin(found)]

In [ ]:
df_hosts['Age_lower'] = df_hosts['Age'].apply(lambda x: str(x).split('-')[0]).astype(float).values
df_hosts['Age_upper'] = df_hosts['Age'].apply(lambda x: str(x).split('-')[1]).astype(float).values

## Kepler vs TESS young planets

## New discoveries

In [ ]:
import astropy.units as u

# TIC*1445
target = {'pl_orbper': (3.272052, 1e-05), #best['b_period'],
          'pl_rade': (5.7, 0.7), #(Rp_samples.median(), Rp_samples.std()),
}

# TOI-6715
target2 = {'pl_orbper': (2.8625896500316537, 3.263991441179825e-06),
 # 'pl_rade': (3.8858759003555825, 0.2871934014216958)
 'pl_rade': (4.75, 0.3)
          }

# TOI-????
target3 = {'pl_orbper': (3.16, 1e-05),
 'pl_rade': (8.67, 0.1)}

# TOI-5973
target4 = {'pl_orbper': (3.16, 1e-05),
 'pl_rade': (2.17026355*u.Rjup.to(u.Rearth),
             2.5e-7*u.Rjup.to(u.Rearth))
          }

# TOI-6940
target5 = {'pl_orbper': (0.3052608, 5e-7),
 'pl_rade': #(2.03448, 0.377961)
           (1.4, 0.17)
          }

# TOI-5169
target6 = {'pl_orbper': (8.0387972, 0.0000287),
 'pl_rade': (2.25604, 0.174709)
          }

# TOI-5734
target7 = {'pl_orbper': (6.1841876, 0.0000080),
 'pl_rade': (2.1, 0.12)
          }

special = [#'toi-5734', #Morello et al. 2026
           'hip67522', #Barber et al. 2024
           'tic434398831' #Vach et al. 2025
          ]

In [ ]:
import re
import seaborn as sb
import numpy as np
import matplotlib.pyplot as pl
import matplotlib.ticker as mticker
from collections import OrderedDict
from matplotlib.patches import Rectangle
pl.rcParams['font.size'] = 14

#####
x = 'pl_orbper'
y = 'pl_rade'
show_target = True
show_contour = True
show_young_planets = True
show_empirical_boundary = True
show_theoretical_boundary = False
show_period_radius_gap = False
porb_lims = (0.2, 300)
rp_lims = (0.6, 35)
figsize = (5, 5)
legend = ' Our Work'
#####

fig, ax = pl.subplots(figsize=figsize)

# contour plot of all planets
if show_contour:
    _ = sb.kdeplot(data=df, 
                   x=x, 
                   y=y,  
                   cmap='gray_r', 
                   # n_levels=8, 
                   log_scale=True, 
                   fill=True, 
                   ax=ax, 
                   zorder=-1
                  )
    
# scatter plot of young planets
if show_young_planets:
    periods = df_planets['pl_orbper']
    radii = df_planets['pl_rade']
    names = df_planets['hostname']
    
    # color depending on age
    for xx,yy,n in zip(periods, radii, names):
        idx = young_hostnames==n
        d = df_hosts[idx].squeeze()
        d = d.iloc[0] if isinstance(d, pd.DataFrame) else d #multiple rows
        if len(d)>0:
            try:
                age = float(str(d.Age).split('-')[1])
                marker = '*' if n in special else 'o'
                ec = 'w' #'k' if n in special else 'w'
                s = 14 if n in special else 9
                zorder = 100 if n in special else 10
                if marker=='*':
                    marker = 'o' if n=='hip67522' and int(xx)==14 else '*'
                    s = 9 if n=='hip67522' and int(xx)==14 else 14
                    # if marker=='*':
                    #     print(f"{n}: P={xx},R={yy}")
                if age<100:
                    print(f"{n}: P={xx},R={yy}")
                    color = 'C3'
                    lbl = 'age<100Myr'
                    # zorder = 12
                    ax.plot(xx, yy, marker, label=lbl, ms=s, mec=ec, c=color, zorder=zorder)
                     # white outline
                    # ax.plot(xx, yy, 'o', c='w', label='_nolegend_', ms=8, zorder=6)
                else:
                    # if marker=='*':
                    #     print(f"{n}: P={xx},R={yy}")
                    if n=='toi-5174':
                        print(f"{n}: P={xx},R={yy}")
                    color = 'C0'
                    zorder -= 1
                    lbl = '100Myr<age<1Gyr'
                    ax.plot(xx, yy, marker, ms=s, label=lbl, mec=ec, c=color, zorder=zorder)
                     # white outline
                    # ax.plot(xx, yy, 'o', c='k', label='_nolegend_', ms=8, zorder=6)
                
            except Exception as e:
                 print(f"Error: {n}\n{e}")

if show_target:
    ec = 'w'
    ax.scatter(target3[x], target3[y], marker='*', s=200, color='C3', edgecolors=ec, label=legend, 
               zorder=10)
    ax.scatter(target[x], target[y], marker='*', s=200, color='C3', edgecolors=ec, #label='TIC*1445b', 
               zorder=10)
    ax.scatter(target2[x], target2[y], marker='*', s=200, color='C3', edgecolors=ec, #label='TOI-6715b', 
               zorder=10)
    ax.scatter(target4[x], target4[y], marker='*', s=200, color='C0', edgecolors=ec, #label='TOI-1355', 
               zorder=10)
    ax.scatter(target5[x], target5[y], marker='*', s=200, color='C0', edgecolors=ec, #label='TOI-6940b', 
               zorder=10)
    ax.scatter(target6[x], target6[y], marker='*', s=200, color='C0', edgecolors=ec, #label='TOI-5169b', 
               zorder=10)
    ax.scatter(target7[x], target7[y], marker='*', s=200, color='C0', edgecolors=ec, #label='TOI-5734', 
               zorder=10)
        
# edit axes
ax.set_xlim(*porb_lims)
ax.set_ylim(*rp_lims)
ax.set_ylabel(r'Planet radius (R$_{\oplus}$)')
ax.set_xlabel('Orbital Period (days)')
ax.tick_params(direction='in', which='both', width=2, bottom=True, top=True, left=True, right=True)
lw=2
ax.spines['left'].set_linewidth(lw)
ax.spines['right'].set_linewidth(lw)
ax.spines['top'].set_linewidth(lw)
ax.spines['bottom'].set_linewidth(lw)
ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())

# Neptune desert
if show_theoretical_boundary:
    xs = np.linspace(0.5, 14, 100)
    ys_lo, ys_hi = theoretical_boundary(xs)
    ax.plot(xs, ys_lo,'k',ls='--',lw=2,zorder=1,alpha=0.5)
    ax.plot(xs, ys_hi,'k',ls='--',lw=2,zorder=1,alpha=0.5)

if show_empirical_boundary:
    xbounds, ybounds = empirical_boundary_linear()
    ax.plot(xbounds, ybounds, c='k', linestyle="-.", lw=2, zorder=0, alpha=0.5)

# period-radius gap
if show_period_radius_gap:
    xmodel = np.linspace(0.1, 100, 101)
    ymodel = vaneylen_model_linear(xmodel)
    ax.plot(xmodel, ymodel, 'k', ls=':')

# add legend
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple

custom_handles = [
    Line2D([0], [0], marker='o', color='C3', ms=8, mec='w', ls='none', label='age<100Myr'),
    Line2D([0], [0], marker='o', color='C0', ms=8, mec='w', ls='none', label='100Myr<age<1Gyr'),
    (Line2D([0], [0], marker='*', color='C3', ms=12, mec=ec, ls='none'),
     Line2D([0], [0], marker='*', color='C0', ms=12, mec=ec, ls='none')),
]

custom_labels = ['age<100Myr', '100Myr<age<1Gyr', legend]

leg = ax.legend(custom_handles, custom_labels,
                handler_map={(tuple): HandlerTuple(ndivide=None)},
                fontsize=12,
                framealpha=1,
                loc='upper right',
                markerfirst=True,
                handletextpad=0.1,
                edgecolor='k')
leg.get_frame().set_facecolor('w')
leg.get_frame().set_facecolor('w')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f'))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f'))

## Radius evolution: Cluster vs Field

In [ ]:
cols = ['hostname','Age_lower','Age_upper','Cluster/','Nplanets']
df_planets2 = pd.merge(df_planets, df_hosts[cols], on='hostname')
df_planets2['age'] = df_planets2[['Age_upper','Age_lower']].mean(1)

In [ ]:
xerr1 = df_planets2.age-df_planets2.Age_lower
xerr2 = df_planets2.Age_upper-df_planets2.age

#####
x = 'age'
y = 'pl_rade'
ylims = (0.5,20)
xlims = (0,1200)
figsize = (5,5)
#####

fig, ax = pl.subplots(figsize=figsize)

x = 'age'
y = 'pl_rade'
idx = df_planets2['Cluster/']=='field'
ax.errorbar(x=df_planets2.loc[idx,x], 
            y=df_planets2.loc[idx,y], 
            xerr=np.c_[[xerr1[idx],xerr2[idx]]],
            yerr=np.c_[df_planets2.pl_radeerr1[idx],df_planets2.pl_radeerr1[idx]].T,
            marker='o', color='C0', label='field',
            ls='')
ax.errorbar(x=df_planets2.loc[~idx,x], 
            y=df_planets2.loc[~idx,y], 
            xerr=np.c_[[xerr1[~idx],xerr2[~idx]]],
            yerr=np.c_[df_planets2.pl_radeerr1[~idx],df_planets2.pl_radeerr1[~idx]].T,
            marker='o', color='C3', label='cluster',
            ls='')
ax.set_xscale("log")
ax.set_yscale("log")

# edit axes
ax.set_xlim(*xlims)
ax.set_ylim(*ylims)
ax.set_ylabel(r'Planet radius (R$_{\oplus}$)')
ax.set_xlabel('Age (Myr)')
ax.tick_params(direction='in', which='both', width=2, bottom=True, top=True, left=True, right=True)
lw=1.5
ax.spines['left'].set_linewidth(lw)
ax.spines['right'].set_linewidth(lw)
ax.spines['top'].set_linewidth(lw)
ax.spines['bottom'].set_linewidth(lw)
ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())

# add labels
dx = 4
alpha = 1
Rearth = 1
ax.axhline(Rearth, 0, 1, c='k', ls='--', alpha=alpha, zorder=9)
ax.text(dx, Rearth+0.05, 'Earth', alpha=alpha)
Rnep = 3.88
ax.axhline(Rnep, 0, 1, c='k', ls='--', alpha=alpha, zorder=9)
ax.text(dx, Rnep+0.4, 'Neptune', alpha=alpha)
Rjup = 11.2
ax.axhline(Rjup, 0, 1, c='k', ls='--', alpha=alpha, zorder=9)
ax.text(dx, Rjup+1.1, 'Jupiter', alpha=alpha)
# add legend
handles, labels = pl.gca().get_legend_handles_labels()
by_label = OrderedDict(zip(labels, handles))
pl.legend(by_label.values(), 
          list(by_label.keys()), 
          fontsize=10,
          framealpha=1,
          # bbox_to_anchor=(1.0, 1.0), 
          # prop=font_prop
         )
from matplotlib.ticker import FormatStrFormatter
ax.xaxis.set_major_formatter(FormatStrFormatter('%.0f'))
ax.yaxis.set_major_formatter(FormatStrFormatter('%.0f'))

# edit axes
ax.tick_params(direction='in', which='both', width=2, bottom=True, top=True, left=True, right=True)
lw=2
ax.spines['left'].set_linewidth(lw)
ax.spines['right'].set_linewidth(lw)
ax.spines['top'].set_linewidth(lw)
ax.spines['bottom'].set_linewidth(lw)
ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())

## K2 vs TESS

In [ ]:
# histogram of host-star ages split by Kepler/K2 vs TESS era
age = 0.5 * (df_hosts['Age_lower'] + df_hosts['Age_upper'])
is_kepler = df_hosts['Kepler/K2 era?'] == 1

age_kepler = age[is_kepler]
age_tess = age[~is_kepler]

fig, ax = pl.subplots(figsize=(5, 5))
# linear
# bins = np.linspace(age.min(), age.max(), 15)
# log
# bins = np.logspace(np.log10(age.min()), np.log10(age.max()), 10)
# Freedman–Diaconis rule  for non-Gaussian, small-N
iqr = np.subtract(*np.percentile(age, [75, 25]))
bw = 2 * iqr * len(age)**(-1/3)  # ~Sturges but outlier-robust

ax.hist(age_kepler, bins=bins, color='C3', alpha=0.6,
        edgecolor='w', label=f'Kepler/K2 (N={len(age_kepler)})')
ax.hist(age_tess, bins=bins, color='C0', alpha=0.6,
        edgecolor='w', label=f'TESS (N={len(age_tess)})')

ax.set_xscale('log')
ax.set_xlabel('Age (Myr)')
ax.set_ylabel('Number of host stars')

# edit axes
ax.tick_params(direction='in', which='both', width=2, bottom=True, top=True, left=True, right=True)
lw=2
ax.spines['left'].set_linewidth(lw)
ax.spines['right'].set_linewidth(lw)
ax.spines['top'].set_linewidth(lw)
ax.spines['bottom'].set_linewidth(lw)
ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())

fig.tight_layout()
fig.savefig('../figures/young_planet_ages_hist.png', bbox_inches='tight', dpi=200)

# Insightful diagnostics for young transiting exoplanets

The cells below join two datasets:

- **NExSci Planetary Systems** (`df_nexsci`) — the *field* population of confirmed transiting/RV planets with default ("self-consistent") parameters, spanning all ages.
- **Age-dated young hosts** (`df_hosts`) — TOIs and Kepler/K2 planets with cluster / association / gyrochronology ages (~1–700 Myr): our *accurate-age* sample.

**Why young planets matter for planet formation & evolution.** Young planets are the **time-zero anchors** of empirical formation/evolution models. The mature field population only records the *end state* after Gyr of thermal contraction, photoevaporation, and dynamical sculpting; the young sample exposes the *initial conditions*. Plotting them together converts static, age-blind correlations — the radius valley, the Neptune desert, the mass–radius relation — into **evolutionary tracks** with a real time axis, which is what theory must reproduce. Each plot below pairs a measurement with the specific model ingredient it constrains.

In [ ]:
# =====================================================================
# Setup: build an accurate young-planet sample (NExSci x age-dated hosts)
# =====================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as pl, seaborn as sb
import astropy.units as u
from pathlib import Path
from scipy.stats import ks_2samp

# --- apply project matplotlib style (rcParams.txt) ---
def _coerce(v):
    s = str(v).strip().strip("'\"")
    if s in ('True', 'False'):
        return s == 'True'
    if s == 'none':
        return 'none'
    try:
        return float(s)
    except ValueError:
        return s
_rc = pd.read_csv('../rcParams.txt')
for _k, _v in zip(_rc['name'], _rc['value']):
    try:
        pl.rcParams[_k] = _coerce(_v)
    except Exception:
        pass
pl.rcParams['font.size'] = 13  # talk-friendly

FIGDIR = Path('../figures'); FIGDIR.mkdir(exist_ok=True)
KEP_C, TESS_C, FIELD_C = 'C3', 'C0', '0.55'

# --- field sample: confirmed transiting/RV planets, default parameters ---
df_nexsci['pl_rade'] = df_nexsci['pl_radj'] * u.Rjup.to(u.Rearth)
_f = ((df_nexsci.pl_orbper > 0) & (df_nexsci.pl_rade > 0) & (df_nexsci.st_mass > 0)
      & df_nexsci.discoverymethod.isin(['Transit', 'Radial Velocity'])
      & (df_nexsci.default_flag == 1))
field = df_nexsci[_f].copy()
field['key'] = field['hostname'].str.lower().str.replace(' ', '')

# --- accurate ages from the age-dated young-host table ---
H = df_hosts.copy()
H['key'] = H['Host'].str.lower().str.replace(' ', '')
H['age_lo'] = H['Age'].astype(str).str.split('-').str[0].astype(float)   # Myr
H['age_hi'] = H['Age'].astype(str).str.split('-').str[1].astype(float)   # Myr
H['age_mid'] = 0.5 * (H.age_lo + H.age_hi)
_amap = H.set_index('key')

# --- young sample = field planets around age-dated hosts ---
young = field[field['key'].isin(H['key'])].copy()
young['age_mid'] = young['key'].map(_amap['age_mid'])        # Myr
young['age_lo'] = young['key'].map(_amap['age_lo'])
young['age_hi'] = young['key'].map(_amap['age_hi'])
young['age_gyr'] = young['age_mid'] / 1e3
young['is_kepler'] = young['key'].map(_amap['Kepler/K2 era?']).eq(1)

print(f'field planets : {len(field)}')
print(f'young planets : {len(young)} around {young.key.nunique()} age-dated hosts')
print(f'  Kepler/K2-era: {int(young.is_kepler.sum())}   TESS-era: {int((~young.is_kepler).sum())}')

def _style(ax, logx=False, logy=False):
    ax.tick_params(direction='in', which='both', top=True, right=True)
    if logx:
        ax.set_xscale('log')
    if logy:
        ax.set_yscale('log')

**1. Period-radius plane.** The grayscale density is the mature field population; young planets are overplotted (Kepler/K2-era red, TESS-era blue). Comparing where young planets fall relative to the empirically calibrated **radius valley** (Van Eylen et al. 2018) and **Neptune desert** boundaries tests whether these features are *primordial* (carved at formation) or *evolved* (sculpted later by atmospheric mass loss). Young planets found *inside* the desert or filling in *above* the valley are smoking-gun evidence that the boundaries are sculpted over time — which is exactly what empirical evolution models must reproduce.

In [ ]:
# Plot 1 - Period-radius plane: where do young planets live?
fig, ax = pl.subplots(figsize=(6, 5))
sb.kdeplot(data=field, x='pl_orbper', y='pl_rade', cmap='Greys',
           fill=True, log_scale=True, levels=8, thresh=0.05, ax=ax, zorder=-1)
ax.scatter(young.pl_orbper[young.is_kepler], young.pl_rade[young.is_kepler],
           s=45, c=KEP_C, edgecolor='w', lw=0.6, zorder=6, label='Young (Kepler/K2 era)')
ax.scatter(young.pl_orbper[~young.is_kepler], young.pl_rade[~young.is_kepler],
           s=45, c=TESS_C, edgecolor='w', lw=0.6, zorder=7, label='Young (TESS era)')
P = np.logspace(np.log10(0.5), np.log10(100), 100)
ax.plot(P, 10**(-0.09 * np.log10(P) + 0.37), 'k--', lw=2, zorder=5,
        label='Radius valley (Van Eylen+18)')
Pd = np.logspace(np.log10(0.5), np.log10(14), 100)
ax.plot(Pd, 14.8 * Pd**-0.33, ':', c='C1', lw=2.5, zorder=5)
ax.plot(Pd, Pd**0.68, ':', c='C1', lw=2.5, zorder=5, label='Neptune desert')
ax.set_xlim(0.5, 200); ax.set_ylim(0.7, 30)
ax.set_xlabel('Orbital period (days)'); ax.set_ylabel(r'Planet radius (R$_\oplus$)')
_style(ax, logx=True, logy=True)
ax.legend(fontsize=8.5, framealpha=1, loc='lower left')
fig.tight_layout(); fig.savefig(FIGDIR / '01_period_radius_young.png', dpi=200, bbox_inches='tight')

**2. The radius distribution.** The bimodal radius distribution with a gap near 1.8 R⊕ (the *radius valley*) is the headline evidence for atmospheric mass loss. The key test: if the valley is **shallower or absent** in the young sample and only deepens with age, then photoevaporation / core-powered mass loss (which operate over ~10 Myr–1 Gyr) are favoured over a purely primordial (formation-imprinted) origin. The shape of the young distribution directly constrains the *timescale* term in evolution models.

In [ ]:
# Plot 2 - The radius distribution: is the radius valley primordial?
fig, ax = pl.subplots(figsize=(6, 5))
bins = np.logspace(np.log10(0.7), np.log10(25), 28)
ax.hist(field.pl_rade, bins=bins, density=True, color=FIELD_C, alpha=0.6, label='Field (all ages)')
ax.hist(young.pl_rade, bins=bins, density=True, histtype='step', color=TESS_C, lw=3, label='Young (<~0.7 Gyr)')
ax.axvspan(1.5, 2.0, color='C1', alpha=0.18, label=r'Radius valley (~1.8 R$_\oplus$)')
ax.set_xlabel(r'Planet radius (R$_\oplus$)'); ax.set_ylabel('Normalised density')
_style(ax, logx=True)
ax.set_xticks([1, 2, 3, 5, 10, 20]); ax.set_xticklabels([1, 2, 3, 5, 10, 20])
ax.legend(fontsize=9, framealpha=1)
fig.tight_layout(); fig.savefig(FIGDIR / '02_radius_distribution.png', dpi=200, bbox_inches='tight')

**3. Cumulative distribution + KS test.** A quantitative, binning-independent comparison of the young vs field radius distributions. A small KS p-value confirms the two are drawn from *different* parent populations — i.e. radii genuinely **evolve** rather than reflecting noise — while the sign of the offset (young planets systematically larger) sets the direction and magnitude any evolution model must reproduce. This turns the visual impression from Plot 2 into a defensible statistical statement.

In [ ]:
# Plot 3 - Cumulative radius distribution + KS test (young vs field)
fig, ax = pl.subplots(figsize=(6, 5))
def _cdf(d):
    x = np.sort(d.dropna().values)
    return x, np.arange(1, len(x) + 1) / len(x)
for d, c, lbl in [(field.pl_rade, FIELD_C, 'Field'), (young.pl_rade, TESS_C, 'Young')]:
    x, y = _cdf(d); ax.plot(x, y, c=c, lw=3, label=lbl)
ks = ks_2samp(young.pl_rade.dropna(), field.pl_rade.dropna())
ax.text(0.05, 0.95, f'KS D={ks.statistic:.2f}\np={ks.pvalue:.1e}',
        transform=ax.transAxes, va='top', fontsize=11,
        bbox=dict(fc='w', ec='0.5'))
ax.set_xlabel(r'Planet radius (R$_\oplus$)'); ax.set_ylabel('Cumulative fraction')
_style(ax, logx=True); ax.legend(fontsize=10, loc='lower right', framealpha=1)
fig.tight_layout(); fig.savefig(FIGDIR / '03_radius_cdf_ks.png', dpi=200, bbox_inches='tight')

**4. Planet radius vs age.** This is the most direct evolutionary test: combining NExSci stellar ages for the field with the *accurate* young-host ages, sub-Neptunes (R<4 R⊕) are predicted to **shrink** over the first ~Gyr as their H/He envelopes cool, contract, and partially escape. The running median is an empirical contraction track; its slope and the age at which it flattens directly calibrate the envelope mass-loss efficiency and the cooling timescale in evolution models. The young points carry the asymmetric age error bars that make this track trustworthy at the youngest ages.

In [ ]:
# Plot 4 - Planet radius vs age: contraction & atmospheric escape
fig, ax = pl.subplots(figsize=(6, 5))
fa = field[(field.st_age > 0) & (field.pl_rade < 4)]
ya = young[young.pl_rade < 4]
ax.scatter(fa.st_age, fa.pl_rade, s=12, c=FIELD_C, alpha=0.35,
           label='Field sub-Neptunes (NExSci ages)')
ax.errorbar(ya.age_gyr, ya.pl_rade,
            xerr=[ya.age_gyr - ya.age_lo / 1e3, ya.age_hi / 1e3 - ya.age_gyr],
            fmt='o', ms=6, c=TESS_C, mec='w', ecolor='0.6', elinewidth=1,
            zorder=5, label='Young (accurate ages)')
allx = np.concatenate([fa.st_age.values, ya.age_gyr.values])
ally = np.concatenate([fa.pl_rade.values, ya.pl_rade.values])
b = np.logspace(np.log10(np.nanmin(allx[allx > 0])), np.log10(np.nanmax(allx)), 9)
mid = np.sqrt(b[:-1] * b[1:])
med = [np.nanmedian(ally[(allx >= b[i]) & (allx < b[i + 1])]) for i in range(len(b) - 1)]
ax.plot(mid, med, 'k-', lw=2.5, zorder=6, label='Running median')
ax.set_xlabel('Age (Gyr)'); ax.set_ylabel(r'Planet radius (R$_\oplus$)')
_style(ax, logx=True); ax.set_ylim(0.7, 4.2); ax.legend(fontsize=9, framealpha=1)
fig.tight_layout(); fig.savefig(FIGDIR / '04_radius_vs_age.png', dpi=200, bbox_inches='tight')

**5. Radius vs insolation.** Atmospheric escape is driven by stellar high-energy (XUV) irradiation, which is orders of magnitude stronger around young, magnetically active stars. Highly irradiated young planets that *still* retain large radii reveal how much primordial atmosphere survives the early high-flux epoch, mapping the boundary of the **photoevaporation desert** before it is fully carved. The flux axis is the physically correct control variable for escape models (more fundamental than period alone).

In [ ]:
# Plot 5 - Radius vs insolation: the photoevaporation desert
fig, ax = pl.subplots(figsize=(6, 5))
fi = field[(field.pl_insol > 0) & (field.pl_rade > 0)]
yi = young[young.pl_insol > 0]
ax.scatter(fi.pl_insol, fi.pl_rade, s=10, c=FIELD_C, alpha=0.3, label='Field')
ax.scatter(yi.pl_insol, yi.pl_rade, s=45, c=TESS_C, ec='w', lw=0.6, zorder=5, label='Young')
ax.axhspan(1.8, 4.0, color='C1', alpha=0.12, label='sub-Neptune regime')
ax.set_xlabel(r'Insolation flux (S$_\oplus$)'); ax.set_ylabel(r'Planet radius (R$_\oplus$)')
_style(ax, logx=True, logy=True); ax.legend(fontsize=9, framealpha=1)
fig.tight_layout(); fig.savefig(FIGDIR / '05_radius_vs_insolation.png', dpi=200, bbox_inches='tight')

**6. Mass-radius diagram.** At fixed mass, young planets should sit **above** the field relation (larger radius, lower bulk density) because their H/He envelopes are still hot and extended and have not yet been stripped. The constant-density guide lines frame the degree of inflation. This anchors the *initial* radius in mass-radius-age models and breaks the degeneracy between envelope mass fraction and age that plagues mature-planet samples.

In [ ]:
# Plot 6 - Mass-radius diagram: young planets are inflated / low-density
fig, ax = pl.subplots(figsize=(6, 5))
fm = field[(field.pl_bmasse > 0) & (field.pl_rade > 0)]
ym = young[young.pl_bmasse > 0]
ax.scatter(fm.pl_bmasse, fm.pl_rade, s=10, c=FIELD_C, alpha=0.3, label='Field')
ax.scatter(ym.pl_bmasse, ym.pl_rade, s=55, c=TESS_C, ec='w', lw=0.6, zorder=5, label='Young')
# constant-density guides (relative to Earth, rho_earth = 5.51 g/cc)
M = np.logspace(-0.5, 3.5, 50)
for rho, lab in [(5.51, r'$\rho_\oplus$'), (1.0, 'water')]:
    ax.plot(M, M**(1/3) * (5.51 / rho)**(1/3), '--', c='0.4', lw=1)
ax.text(2e3, (2e3)**(1/3), r'$\rho_\oplus$', fontsize=9, c='0.4')
ax.set_xlabel(r'Planet mass (M$_\oplus$)'); ax.set_ylabel(r'Planet radius (R$_\oplus$)')
_style(ax, logx=True, logy=True); ax.set_ylim(0.7, 30); ax.legend(fontsize=10, framealpha=1, loc='upper left')
fig.tight_layout(); fig.savefig(FIGDIR / '06_mass_radius.png', dpi=200, bbox_inches='tight')

**7. Planet radius vs host-star mass.** Tests whether planet size scales with stellar mass (more massive disks and higher XUV output produce different outcomes). Equally important, overlaying the two samples checks that young and field hosts share the same stellar-mass coverage — a guard against the **selection bias** that age-dated young samples are dominated by particular spectral types, which would otherwise masquerade as an evolutionary trend.

In [ ]:
# Plot 7 - Planet radius vs host-star mass
fig, ax = pl.subplots(figsize=(6, 5))
ax.scatter(field.st_mass, field.pl_rade, s=10, c=FIELD_C, alpha=0.3, label='Field')
ax.scatter(young.st_mass, young.pl_rade, s=45, c=TESS_C, ec='w', lw=0.6, zorder=5, label='Young')
ax.set_xlabel(r'Host-star mass (M$_\odot$)'); ax.set_ylabel(r'Planet radius (R$_\oplus$)')
_style(ax, logy=True); ax.legend(fontsize=10, framealpha=1)
fig.tight_layout(); fig.savefig(FIGDIR / '07_radius_vs_stellarmass.png', dpi=200, bbox_inches='tight')

**8. Orbital-period distribution.** Young samples are biased toward short periods (transits are easier to detect over short photometric baselines), so the bulk shapes mostly reflect selection. The *informative* signal is a genuine deficit: a relative lack of **ultra-short-period** (<1 d) planets when young would indicate USPs are *produced later* by tidal decay and high-eccentricity migration rather than formed in situ — a direct test of migration channels in evolution models.

In [ ]:
# Plot 8 - Orbital-period distribution: young vs field
fig, ax = pl.subplots(figsize=(6, 5))
bins = np.logspace(np.log10(0.5), np.log10(200), 26)
ax.hist(field.pl_orbper, bins=bins, density=True, color=FIELD_C, alpha=0.6, label='Field')
ax.hist(young.pl_orbper, bins=bins, density=True, histtype='step', color=TESS_C, lw=3, label='Young')
ax.set_xlabel('Orbital period (days)'); ax.set_ylabel('Normalised density')
_style(ax, logx=True); ax.legend(fontsize=10, framealpha=1)
fig.tight_layout(); fig.savefig(FIGDIR / '08_period_distribution.png', dpi=200, bbox_inches='tight')

**9. Age distribution of the host sample.** Log-spaced bins are used deliberately: ages span ~2 decades and their uncertainties are roughly *fractional* (a fixed percentage of the age), so equal ratios — not equal differences — are the physically meaningful unit. Splitting Kepler/K2-era vs TESS-era shows how TESS has pushed age-dated detections into the youngest (<100 Myr) regime, exactly where evolutionary signals (contraction, escape) are strongest and where formation models are least constrained.

In [ ]:
# Plot 9 - Age distribution of the age-dated host sample (log bins)
fig, ax = pl.subplots(figsize=(6, 5))
ages = H.dropna(subset=['age_mid'])
kep = ages['Kepler/K2 era?'] == 1
bins = np.logspace(np.log10(ages.age_mid.min()), np.log10(ages.age_mid.max()), 12)
ax.hist(ages.age_mid[kep], bins=bins, color=KEP_C, alpha=0.7, edgecolor='w',
        label=f'Kepler/K2 era (N={int(kep.sum())})')
ax.hist(ages.age_mid[~kep], bins=bins, color=TESS_C, alpha=0.7, edgecolor='w',
        label=f'TESS era (N={int((~kep).sum())})')
ax.set_xlabel('Age (Myr)'); ax.set_ylabel('Number of hosts')
_style(ax, logx=True); ax.legend(fontsize=10, framealpha=1)
fig.tight_layout(); fig.savefig(FIGDIR / '09_age_distribution.png', dpi=200, bbox_inches='tight')

**10. Radius-age trends split by irradiation.** Splitting into hot (P<10 d) and warm (P>10 d) planets isolates the *irradiation dependence* of mass loss. If only the **hot** sub-Neptunes shrink with age while warm ones stay puffy, atmospheric escape is irradiation-driven (photoevaporation / core-powered mass loss) — precisely the behaviour empirical evolution models encode. A flat warm track also rules out age systematics as the cause of the hot-planet shrinkage.

In [ ]:
# Plot 10 - Radius-age trends split by irradiation (hot vs warm)
fig, ax = pl.subplots(figsize=(6, 5))
comb = pd.concat([
    field[(field.st_age > 0) & (field.pl_rade < 4)][['pl_orbper', 'pl_rade', 'st_age']]
        .rename(columns={'st_age': 'age_gyr'}),
    young[young.pl_rade < 4][['pl_orbper', 'pl_rade', 'age_gyr']],
], ignore_index=True)
hot = comb.pl_orbper < 10
for sel, c, lbl in [(hot, 'C3', 'Hot (P<10 d)'), (~hot, 'C0', 'Warm (P>10 d)')]:
    d = comb[sel]
    ax.scatter(d.age_gyr, d.pl_rade, s=14, c=c, alpha=0.45, label=lbl)
    b = np.logspace(np.log10(max(d.age_gyr.min(), 1e-3)), np.log10(d.age_gyr.max()), 7)
    mid = np.sqrt(b[:-1] * b[1:])
    med = [np.nanmedian(d.pl_rade[(d.age_gyr >= b[i]) & (d.age_gyr < b[i + 1])]) for i in range(len(b) - 1)]
    ax.plot(mid, med, '-', c=c, lw=3, zorder=5)
ax.set_xlabel('Age (Gyr)'); ax.set_ylabel(r'Planet radius (R$_\oplus$)')
_style(ax, logx=True); ax.set_ylim(0.7, 4.2); ax.legend(fontsize=10, framealpha=1)
fig.tight_layout(); fig.savefig(FIGDIR / '10_radius_age_irradiation.png', dpi=200, bbox_inches='tight')

# Unique features of our targets ("Our Work")

The seven `target` planets defined above are placed in context against **other young planets** (`young`) and their **mature field counterparts** (`field`). Four groups stand out:

- **Neptune-desert dwellers** — TIC*1445, TOI-6715, TOI-? (R ≈ 4.7–8.7 R⊕ at P ≈ 3 d) land *inside* the Neptune desert: ~80th radius percentile but only ~12th period percentile of the young sample. Hot Neptunes this large are rare at any age and are exactly the objects mass-loss models predict should be transient — catching them young constrains how fast the desert is carved.
- **Young ultra-short-period planet** — TOI-6940 (P = 0.31 d) is shorter-period than *every* other young planet here (0th percentile); USPs are <1% of the field, so a young one tests whether they form in situ or migrate in later.
- **Inflated young giant** — TOI-5973 (≈24 R⊕ = 2.17 R$_{\rm Jup}$) sits at the 100th radius percentile, probing envelope inflation at early ages.
- **Radius-valley keepers** — TOI-5169 & TOI-5734 (~2.1–2.3 R⊕) sit just *above* the valley, retaining envelopes their mature analogues have largely lost.

In [ ]:
# ---- Build the "Our Work" target sample from the target dicts above ----
import astropy.units as u

_RJ = u.Rjup.to(u.Rearth)
TCOL = {'<100 Myr': 'C3', '0.1-1 Gyr': 'C0'}   # red = youngest, blue = older young

_rows = [('TIC*1445', target,  '<100 Myr'),
         ('TOI-6715', target2, '<100 Myr'),
         ('TOI-? (t3)', target3, '<100 Myr'),
         ('TOI-5973', target4, '0.1-1 Gyr'),
         ('TOI-6940', target5, '0.1-1 Gyr'),
         ('TOI-5169', target6, '0.1-1 Gyr'),
         ('TOI-5734', target7, '0.1-1 Gyr')]
targets = pd.DataFrame([{
    'name': nm, 'ageclass': ac,
    'P': d['pl_orbper'][0], 'Perr': d['pl_orbper'][1],
    'R': d['pl_rade'][0], 'Rerr': d['pl_rade'][1],
} for nm, d, ac in _rows])

# placement vs empirical relations
targets['R_valley'] = 10**(-0.09 * np.log10(targets.P) + 0.37)          # Van Eylen+18
targets['above_valley'] = targets.R > targets.R_valley
_lo, _hi = targets.P**0.68, 14.8 * targets.P**-0.33                     # Neptune desert (linear)
targets['in_desert'] = (targets.R > _lo) & (targets.R < _hi) & (targets.P < 14)

def _pctile(ref, v):
    a = np.asarray(ref, float); a = a[np.isfinite(a)]
    return 100 * (a < v).mean()
for col, ref in [('Rpct_young', young.pl_rade), ('Ppct_young', young.pl_orbper),
                 ('Rpct_field', field.pl_rade), ('Ppct_field', field.pl_orbper)]:
    src = targets.R if col.startswith('R') else targets.P
    targets[col] = [_pctile(ref, v) for v in src]

print(targets[['name', 'P', 'R', 'R_valley', 'above_valley', 'in_desert',
               'Rpct_young', 'Ppct_young']].round(2).to_string(index=False))

**11. Targets in the period-radius plane.** Stars mark our targets (red = <100 Myr, blue = 0.1–1 Gyr) over the field density (grey) and other young planets. The shaded **Neptune desert** and dashed **radius valley** make the unique placements obvious: three targets sit deep inside the desert, one is an inflated giant above it, and TOI-6940 is pinned to the short-period wall. These are precisely the configurations that empirically test whether the desert and valley are sculpted by evolution.

In [ ]:
# Plot 11 - Our targets in the period-radius plane (desert + valley)
fig, ax = pl.subplots(figsize=(6.5, 5.5))
sb.kdeplot(data=field, x='pl_orbper', y='pl_rade', cmap='Greys',
           fill=True, log_scale=True, levels=8, thresh=0.05, ax=ax, zorder=-1)
ax.scatter(young.pl_orbper, young.pl_rade, s=16, c='0.5', alpha=0.7, zorder=2,
           label='Other young planets')
# Neptune desert (shaded) + radius valley
Pd = np.logspace(np.log10(0.3), np.log10(14.5), 200)
lo, hi = Pd**0.68, 14.8 * Pd**-0.33
m = hi > lo
ax.fill_between(Pd[m], lo[m], hi[m], color='C1', alpha=0.13, zorder=0, label='Neptune desert')
Pv = np.logspace(np.log10(0.2), np.log10(100), 100)
ax.plot(Pv, 10**(-0.09 * np.log10(Pv) + 0.37), 'k--', lw=1.8, zorder=1, label='Radius valley')
# targets
for _, t in targets.iterrows():
    ax.errorbar(t.P, t.R, xerr=t.Perr, yerr=t.Rerr, fmt='*', ms=19,
                mfc=TCOL[t.ageclass], mec='k', ecolor='k', elinewidth=1, zorder=10)
    ax.annotate(t['name'], (t.P, t.R), xytext=(5, 4), textcoords='offset points',
                fontsize=8.5, fontweight='bold', zorder=11)
ax.set_xlim(0.2, 300); ax.set_ylim(0.6, 35)
ax.set_xlabel('Orbital period (days)'); ax.set_ylabel(r'Planet radius (R$_\oplus$)')
_style(ax, logx=True, logy=True)
ax.legend(fontsize=9, loc='lower left', framealpha=1)
fig.tight_layout(); fig.savefig(FIGDIR / '11_targets_period_radius.png', dpi=200, bbox_inches='tight')

**12. Marginal distributions.** Projecting onto radius (left) and period (right) shows *how far into the tails* each target lies relative to the young and field populations. The desert dwellers populate the under-dense ~4–9 R⊕ radius range, while TOI-6940's period line sits beyond the short-period edge of both distributions — turning the visual rarity into a population-level statement.

In [ ]:
# Plot 12 - Where targets fall in the radius & period distributions
fig, axes = pl.subplots(1, 2, figsize=(11, 4.5))
rb = np.logspace(np.log10(0.6), np.log10(35), 30)
axes[0].hist(field.pl_rade, bins=rb, density=True, color='0.7', alpha=0.6, label='Field')
axes[0].hist(young.pl_rade, bins=rb, density=True, histtype='step', color=TESS_C, lw=2.5, label='Young')
axes[0].axvspan(1.5, 2.0, color='k', alpha=0.12)
axes[0].set_xlabel(r'Planet radius (R$_\oplus$)'); axes[0].set_ylabel('Normalised density')
pb = np.logspace(np.log10(0.2), np.log10(300), 30)
axes[1].hist(field.pl_orbper, bins=pb, density=True, color='0.7', alpha=0.6, label='Field')
axes[1].hist(young.pl_orbper, bins=pb, density=True, histtype='step', color=TESS_C, lw=2.5, label='Young')
axes[1].set_xlabel('Orbital period (days)'); axes[1].set_ylabel('Normalised density')
for ax, col in zip(axes, ['R', 'P']):
    for _, t in targets.iterrows():
        ax.axvline(t[col], color=TCOL[t.ageclass], lw=1.6, alpha=0.9, zorder=5)
    ax.set_xscale('log'); ax.tick_params(direction='in', which='both', top=True, right=True)
    ax.legend(fontsize=9, framealpha=1)
fig.tight_layout(); fig.savefig(FIGDIR / '12_targets_marginals.png', dpi=200, bbox_inches='tight')

**13. Quantified uniqueness.** For each target, its radius and period percentile *within the young sample* are drawn as a dumbbell. Targets with a high radius percentile but a low period percentile (the desert hot-Neptunes) span the widest gaps; TOI-6940 collapses to the left edge (shortest period of any young planet). This single figure ranks which targets are the most unusual relative to known young planets — the strongest cases for follow-up and for anchoring formation/evolution models.

In [ ]:
# Plot 13 - Quantified uniqueness: target percentiles within the young sample
order = targets.sort_values('Ppct_young').reset_index(drop=True)
fig, ax = pl.subplots(figsize=(7, 5))
yv = np.arange(len(order))
ax.hlines(yv, order.Ppct_young, order.Rpct_young, color='0.7', lw=2, zorder=1)
ax.scatter(order.Ppct_young, yv, s=95, c='C2', zorder=3, edgecolor='w', label='Period percentile')
ax.scatter(order.Rpct_young, yv, s=95, c='C4', zorder=3, edgecolor='w', label='Radius percentile')
ax.axvline(50, color='k', ls=':', alpha=0.5)
ax.set_yticks(yv); ax.set_yticklabels(order.name)
ax.set_xlim(-3, 103); ax.set_xlabel('Percentile within the young-planet sample (%)')
ax.tick_params(direction='in', which='both', top=True, right=True)
ax.legend(fontsize=9, loc='lower right', framealpha=1)
fig.tight_layout(); fig.savefig(FIGDIR / '13_targets_percentiles.png', dpi=200, bbox_inches='tight')